Finetuned- LACO BERT

In [ ]:
import os, re, string, math, random, inspect, collections, warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from collections import Counter
from tqdm.auto import tqdm
from datasets import load_dataset, Dataset, DatasetDict
import transformers
from transformers import AutoTokenizer, AutoModel, AutoModelForQuestionAnswering, TrainingArguments, Trainer, default_data_collator
warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| transformers", transformers.__version__)


DATA_DIR = "/kaggle/input/datasets/**/2250-dataset"
training_file = f"{DATA_DIR}/train_dataset.json"
validation_file = f"{DATA_DIR}/validate_dataset.json"
test_file = f"{DATA_DIR}/test_dataset.json"
OUT_DIR = "/kaggle/working"
MODEL_NAME = "bert-large-uncased-whole-word-masking-finetuned-squad"
batch_size = 16
EPOCHS = 4
LR = 2e-5
MAX_LENGTH = 480
DOC_STRIDE = 128
FREEZE_BASE = False
TOP_RATIO = 0.25
MIN_TOP, MAX_TOP = 3, 8
EXPAND_WINDOW = 1
KEEP_LEAD = True
MAX_SENT_CHARS = 300
MIN_SENTS_TO_PRUNE = 4
PRUNE_ONLY_IF_LONG = False
TOKEN_MARGIN = 4


dataset = load_dataset("json", data_files={"train": training_file,
                                           "validation": validation_file,
                                           "test": test_file}, field="data")
print(dataset)

raw = {k: dataset[k].to_pandas() for k in ["train", "validation", "test"]}
for k, d in raw.items():
    d["answer_start"] = d["answer"].apply(lambda a: int(a["answer_start"]))
    d["answer_end"]   = d["answer"].apply(lambda a: int(a["answer_end"]))        # inclusive
    d["gold_text"]    = [c[s:e + 1] for c, s, e in zip(d["context"], d["answer_start"], d["answer_end"])]
    d["context_words"] = d["context"].str.split().str.len()
    print(f"{k:<11}: {len(d):>5} rows | unique contexts {d['context'].nunique():>5} | "
          f"context words median {d['context_words'].median():.0f} | gold span words median {d['gold_text'].str.split().str.len().median():.0f}")
raw["train"][["question", "gold_text"]].head(3)

_SENT_BOUNDARY = re.compile(
    r"(?<=[.!?])(?<!\bSec\.)(?<!\bNo\.)(?<!\bNos\.)(?<!\bCl\.)(?<!\bs\.)(?<!\bArt\.)(?<!\bi\.e\.)(?<!\be\.g\.)"
    r"(?<!\bvs\.)(?<!\bMr\.)(?<!\bDr\.)\s+(?=[A-Z0-9\"“'(\[])"
)
_SEMI = re.compile(r"(?<=;)\s+")

def _split_by(text, s, e, rx):
    out, cur = [], s
    for m in rx.finditer(text, s, e):
        if m.start() > cur: out.append((cur, m.start()))
        cur = m.end()
    if cur < e: out.append((cur, e))
    return out

def segment_sentences(text: str):
    spans = []
    for s, e in _split_by(text, 0, len(text), _SENT_BOUNDARY):
        if e - s > MAX_SENT_CHARS:
            spans.extend(_split_by(text, s, e, _SEMI))        # split enumerations "(a) ...; (b) ...;"
        else:
            spans.append((s, e))
    clean = []
    for s, e in spans:                                         # trim whitespace so offsets stay tight
        while s < e and text[s].isspace(): s += 1
        while e > s and text[e - 1].isspace(): e -= 1
        if e > s: clean.append((s, e))
    return clean

_ex = raw["test"].loc[0, "context"]
for i, (s, e) in enumerate(segment_sentences(_ex)):
    print(i, "|", _ex[s:e][:110])

all_contexts = pd.concat([raw[k]["context"] for k in raw]).unique().tolist()
seg_cache = {c: segment_sentences(c) for c in all_contexts}
n_sents = pd.Series([len(v) for v in seg_cache.values()])
print(f"Unique contexts: {len(all_contexts):,} | sentences/context mean={n_sents.mean():.1f} median={n_sents.median():.0f} max={n_sents.max()}")


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
emb_model = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()

@torch.no_grad()
def embed(texts, bs=32):
    texts = list(texts)
    if not texts:
        return np.zeros((0, emb_model.config.hidden_size), dtype="float32")
    order = np.argsort([len(t) for t in texts])
    out = np.zeros((len(texts), emb_model.config.hidden_size), dtype="float32")
    dev = next(emb_model.parameters()).device
    for i in tqdm(range(0, len(texts), bs), disable=len(texts) <= 256):
        idx = order[i:i + bs]
        inp = tokenizer([texts[j] for j in idx], return_tensors="pt", truncation=True, max_length=512, padding=True).to(dev)
        h = emb_model(**inp).last_hidden_state
        m = inp["attention_mask"].unsqueeze(-1).to(h.dtype)
        v = (h * m).sum(1) / m.sum(1).clamp(min=1)
        out[idx] = torch.nn.functional.normalize(v, dim=-1).float().cpu().numpy()
    return out

class LACO:
    def __init__(self, tokenizer, embed_fn, max_len):
        self.tok = tokenizer
        self.embed_fn = embed_fn
        self.max_len = max_len
        self.sent_vec = {}
        self.token_counts = {}

    def cache_sentence_vectors(self, sentences):
        new = [s for s in dict.fromkeys(sentences) if s not in self.sent_vec]
        if new:
            self.sent_vec.update(dict(zip(new, self.embed_fn(new))))

    def n_tokens(self, sentence):
        if sentence not in self.token_counts:
            self.token_counts[sentence] = len(self.tok.tokenize(sentence))
        return self.token_counts[sentence]

    def score_sentences(self, question_vector, sentences):
        self.cache_sentence_vectors(sentences)
        vectors = np.stack([self.sent_vec[s] for s in sentences])
        scores = vectors @ question_vector
        return scores

    def optimize(self, question, question_vector, context, spans):
        sentences = [context[s:e] for s, e in spans]
        if not sentences:
            return {"text": context, "kept": [], "top": [], "new_off": {}, "score": np.array([]), "n_sents": 0}
        scores = self.score_sentences(question_vector, sentences)
        budget = max(32, self.max_len - len(self.tok.tokenize(question)) - 3 - TOKEN_MARGIN)
        fits = sum(self.n_tokens(s) for s in sentences) <= budget
        keep_all = len(sentences) <= MIN_SENTS_TO_PRUNE or (PRUNE_ONLY_IF_LONG and fits)
        k = len(sentences) if keep_all else min(len(sentences), int(np.clip(math.ceil(TOP_RATIO * len(sentences)), MIN_TOP, MAX_TOP)))
        top = sorted(np.argsort(-scores)[:k].astype(int).tolist())
        candidates = set(top)
        if KEEP_LEAD:
            candidates.add(0)
        for i in top:
            for j in range(max(0, i - EXPAND_WINDOW), min(len(sentences), i + EXPAND_WINDOW + 1)):
                candidates.add(j)
        priority = top + [j for j in sorted(candidates) if j not in top]
        kept, used, seen = [], 0, set()
        for i in priority:
            norm = re.sub(r"\W+", " ", sentences[i].lower()).strip()
            if norm in seen:
                continue
            count = self.n_tokens(sentences[i])
            if used + count <= budget or not kept:
                kept.append(i)
                used += count
                seen.add(norm)
        kept.sort()
        parts, new_offsets, pos = [], {}, 0
        for i in kept:
            new_offsets[i] = pos
            parts.append(sentences[i])
            pos += len(sentences[i]) + 1
        return {"text": " ".join(parts), "kept": kept, "top": top, "new_off": new_offsets, "score": scores, "n_sents": len(sentences)}

def remap_span(a, b, spans, result):
    hit = [i for i, (s, e) in enumerate(spans) if e > a and s < b]
    if not hit:
        return None
    kept = set(result["kept"])
    if any(j not in kept for j in range(hit[0], hit[-1] + 1)):
        return None
    start_sentence = hit[0]
    end_sentence = hit[-1]
    start_char = spans[start_sentence][0]
    end_char = spans[end_sentence][0]
    return (result["new_off"][start_sentence] + max(0, a - start_char), result["new_off"][end_sentence] + min(b, spans[end_sentence][1]) - end_char)

laco = LACO(tokenizer, embed, MAX_LENGTH)


all_sents = [c[s:e] for c in all_contexts for s, e in seg_cache[c]]
print(f"Embedding {len(set(all_sents)):,} unique sentences")
laco.cache_sentence_vectors(all_sents)
qvec = {}
for split in raw:
    print(f"Embedding {split} questions")
    qvec[split] = embed(raw[split]["question"].tolist())
emb_model.to("cpu")
if torch.cuda.is_available():
    torch.cuda.empty_cache()


def apply_laco(d, qv, name):
    opt, ns, ne, cov, kept_n = [], [], [], [], []
    for i, row in d.reset_index(drop=True).iterrows():
        spans = seg_cache[row["context"]]
        res   = laco.optimize(row["question"], qv[i], row["context"], spans)
        rm    = remap_span(row["answer_start"], row["answer_end"] + 1, spans, res)
        opt.append(res["text"]); cov.append(rm is not None)
        ns.append(rm[0] if rm else -1); ne.append(rm[1] - 1 if rm else -1)     # new_end inclusive
        kept_n.append(len(res["kept"]))
    out = d.reset_index(drop=True).copy()
    out["opt_context"], out["opt_start"], out["opt_end"] = opt, ns, ne
    out["covered"], out["n_kept"] = cov, kept_n
    print(f"[{name:<10}] answer coverage {np.mean(cov)*100:6.2f}% | "
          f"sentences retained {np.mean(kept_n):.1f}/{out['context'].map(lambda c: len(seg_cache[c])).mean():.1f}")
    return out

opt = {k: apply_laco(raw[k], qvec[k], k) for k in raw}


def show_laco_example(split="test", idx=0):
    row = opt[split].iloc[idx]
    spans = seg_cache[row["context"]]
    result = laco.optimize(row["question"], qvec[split][idx], row["context"], spans)
    print("Question:", row["question"])
    print("Gold answer:", row["gold_text"])
    for i, (start, end) in enumerate(spans):
        status = ("SELECTED" if i in result["top"] else "NOT SELECTED") + (" / KEPT" if i in result["kept"] else "")
        print(status, round(float(result["score"][i]), 4), row["context"][start:end][:160])
    print("Answer retained:", row["covered"])

show_laco_example()


def to_hf(d, only_covered):
    if only_covered: d = d[d["covered"]]
    return Dataset.from_dict({
        "id": d["id"].tolist(), "title": d["title"].tolist(),
        "context": d["opt_context"].tolist(), "question": d["question"].tolist(),
        "answer": [{"answer_start": int(s), "answer_end": int(e)} for s, e in zip(d["opt_start"], d["opt_end"])],
    })

laco_dataset = DatasetDict({
    "train":      to_hf(opt["train"],      True),
    "validation": to_hf(opt["validation"], True),
    "test":       to_hf(opt["test"],       False),
})
print(laco_dataset)

def prepare_train_samples(examples):
    tokenized_examples = tokenizer(
        examples["question"], examples["context"],
        truncation="only_second", max_length=MAX_LENGTH, stride=DOC_STRIDE,
        return_overflowing_tokens=True, return_offsets_mapping=True, padding="max_length",
    )
    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized_examples.pop("offset_mapping")
    tokenized_examples["start_positions"], tokenized_examples["end_positions"] = [], []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized_examples["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        sequence_ids = tokenized_examples.sequence_ids(i)
        answer = examples["answer"][sample_mapping[i]]
        start_char_index, end_char_index = answer["answer_start"], answer["answer_end"] + 1

        token_start_index = 0
        while sequence_ids[token_start_index] != 1: token_start_index += 1
        token_end_index = len(input_ids) - 1
        while sequence_ids[token_end_index] != 1: token_end_index -= 1

        if not (offsets[token_start_index][0] <= start_char_index and offsets[token_end_index][1] >= end_char_index):
            tokenized_examples["start_positions"].append(cls_index)
            tokenized_examples["end_positions"].append(cls_index)
        else:
            while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char_index:
                token_start_index += 1
            tokenized_examples["start_positions"].append(token_start_index - 1)
            while offsets[token_end_index][1] >= end_char_index:
                token_end_index -= 1
            tokenized_examples["end_positions"].append(token_end_index + 1)
    return tokenized_examples

tokenized_dataset = DatasetDict({
    k: laco_dataset[k].map(prepare_train_samples, batched=True, remove_columns=laco_dataset[k].column_names)
    for k in ["train", "validation"]})
print(tokenized_dataset)

model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)
if FREEZE_BASE:
    for p in model.base_model.parameters(): p.requires_grad = False
print("Trainable parameters:", f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

_ta  = inspect.signature(TrainingArguments.__init__).parameters
_eval_key = "eval_strategy" if "eval_strategy" in _ta else "evaluation_strategy"
args = TrainingArguments(
    "laco-test-squad", **{_eval_key: "epoch"}, save_strategy="no", report_to="none",
    learning_rate=LR, per_device_train_batch_size=batch_size, per_device_eval_batch_size=batch_size,
    num_train_epochs=EPOCHS, weight_decay=0.01, logging_dir="./logs", logging_steps=10,
    fp16=torch.cuda.is_available(), seed=SEED,
)
_tok_key = "processing_class" if "processing_class" in inspect.signature(Trainer.__init__).parameters else "tokenizer"
trainer = Trainer(model, args,
                  train_dataset=tokenized_dataset["train"], eval_dataset=tokenized_dataset["validation"],
                  data_collator=default_data_collator, **{_tok_key: tokenizer})
trainer.train()

trainer.save_model(f"{OUT_DIR}/LACOLegalTrained")
logs = pd.DataFrame(trainer.state.log_history)
tr = logs.dropna(subset=["loss"]); ev = logs.dropna(subset=["eval_loss"])
plt.figure(figsize=(8, 4))
plt.plot(tr["step"], tr["loss"], label="Train loss", color="#4C72B0", alpha=.8)
plt.plot(ev["step"], ev["eval_loss"], "o--", label="Val loss", color="#DD8452")
plt.xlabel("Step"); plt.ylabel("Loss"); plt.title("LACO + BERT-Large – Loss"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/laco_loss_curve.png", dpi=150); plt.show()

def prepare_test_samples(examples):
    tokenized_examples = tokenizer(
        examples["question"], examples["context"],
        truncation="only_second", max_length=MAX_LENGTH, stride=DOC_STRIDE,
        return_overflowing_tokens=True, return_offsets_mapping=True, padding="max_length",
    )
    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    tokenized_examples["example_id"] = []
    for i in range(len(tokenized_examples["input_ids"])):
        sequence_ids = tokenized_examples.sequence_ids(i)
        tokenized_examples["example_id"].append(examples["id"][sample_mapping[i]])
        tokenized_examples["offset_mapping"][i] = [
            (o if sequence_ids[k] == 1 else None) for k, o in enumerate(tokenized_examples["offset_mapping"][i])]
    return tokenized_examples

test_samples = laco_dataset["test"].map(prepare_test_samples, batched=True,
                                        remove_columns=laco_dataset["test"].column_names)
raw_predictions = trainer.predict(test_samples.remove_columns(["example_id", "offset_mapping"]))
print("Test windows:", len(test_samples), "for", len(laco_dataset["test"]), "questions")

def postprocess_qa_predictions(examples, samples, raw_preds, best_size=20, max_answer_length=100):
    all_start_logits, all_end_logits = raw_preds
    id2idx = {k: i for i, k in enumerate(examples["id"])}
    per_example = collections.defaultdict(list)
    for i, sample in enumerate(samples):
        per_example[id2idx[sample["example_id"]]].append(i)

    predictions = collections.OrderedDict()
    for ex_i, example in enumerate(tqdm(examples)):
        valid, context = [], example["context"]
        for si in per_example[ex_i]:
            sl, el, offs = all_start_logits[si], all_end_logits[si], samples[si]["offset_mapping"]
            for s_idx in np.argsort(sl)[-1:-best_size - 1:-1].tolist():
                for e_idx in np.argsort(el)[-1:-best_size - 1:-1].tolist():
                    if (s_idx >= len(offs) or e_idx >= len(offs) or offs[s_idx] is None or offs[e_idx] is None):
                        continue
                    if e_idx < s_idx or e_idx - s_idx + 1 > max_answer_length:
                        continue
                    valid.append({"score": sl[s_idx] + el[e_idx], "text": context[offs[s_idx][0]: offs[e_idx][1]]})
        predictions[example["id"]] = max(valid, key=lambda x: x["score"])["text"] if valid else ""
    return predictions

final_predictions = postprocess_qa_predictions(laco_dataset["test"], test_samples, raw_predictions.predictions)


def normalize_answer(text):
    text = text.lower()
    text = "".join(ch for ch in text if ch not in set(string.punctuation))
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())

def token_prf(prediction, reference):
    pred_tokens = normalize_answer(prediction).split()
    ref_tokens = normalize_answer(reference).split()
    common = sum((Counter(pred_tokens) & Counter(ref_tokens)).values())
    if not pred_tokens or not ref_tokens or common == 0:
        return 0.0, 0.0, 0.0
    precision = common / len(pred_tokens)
    recall = common / len(ref_tokens)
    f1 = 2 * precision * recall / (precision + recall)
    return precision, recall, f1

test_df = opt["test"].reset_index(drop=True)
predictions = [final_predictions[row_id] for row_id in test_df["id"]]
references = test_df["gold_text"].tolist()
rows = []
for question, prediction, reference, covered in zip(test_df["question"], predictions, references, test_df["covered"]):
    precision, recall, f1 = token_prf(prediction, reference)
    rows.append({"question": question, "gold_answer": reference, "predicted_answer": prediction, "answer_retained": bool(covered), "exact_match": int(normalize_answer(prediction) == normalize_answer(reference)), "precision": precision, "recall": recall, "f1": f1})
results_df = pd.DataFrame(rows)
summary = pd.DataFrame({"All test examples": results_df[["exact_match", "precision", "recall", "f1"]].mean() * 100, "LACO-retained examples": results_df.loc[results_df.answer_retained, ["exact_match", "precision", "recall", "f1"]].mean() * 100}).round(2)
print(summary)
print(f"Answer retained by LACO: {results_df.answer_retained.mean()*100:.2f}%")
print(f"Exact Match: {results_df.exact_match.mean()*100:.2f}%")
print(f"Token-level F1: {results_df.f1.mean()*100:.2f}%")
results_df.to_csv(f"{OUT_DIR}/bert_laco_test_predictions.csv", index=False)
trainer.save_model(f"{OUT_DIR}/BERT_LACO_LegalQA")
tokenizer.save_pretrained(f"{OUT_DIR}/BERT_LACO_LegalQA")
plt.figure(figsize=(6, 4))
plt.bar(["Exact Match", "Token F1"], [results_df.exact_match.mean()*100, results_df.f1.mean()*100])
plt.ylabel("Score (%)")
plt.ylim(0, 100)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/bert_laco_em_f1.png", dpi=300)
plt.show()


@torch.no_grad()
def laco_answer(question, context):
    spans = segment_sentences(context)
    laco.cache_sentence_vectors([context[s:e] for s, e in spans])
    res = laco.optimize(question, embed([question])[0], context, spans)
    enc = tokenizer(question, res["text"], truncation="only_second", max_length=MAX_LENGTH,
                    return_offsets_mapping=True, return_tensors="pt")
    offs = enc.pop("offset_mapping")[0].numpy(); seq = np.array([s == 1 for s in enc.sequence_ids(0)])
    mdl = trainer.model.eval()
    out = mdl(**{k: v.to(mdl.device) for k, v in enc.items()})
    s_l = np.where(seq, out.start_logits[0].cpu().numpy(), -1e9); e_l = np.where(seq, out.end_logits[0].cpu().numpy(), -1e9)
    best, best_sc = None, -1e18
    for a in np.argsort(-s_l)[:20]:
        for z in np.argsort(-e_l)[:20]:
            if a <= z < a + 100 and seq[a] and seq[z] and s_l[a] + e_l[z] > best_sc:
                best, best_sc = (a, z), s_l[a] + e_l[z]
    return (res["text"][offs[best[0]][0]: offs[best[1]][1]] if best else ""), res["text"]

Finetuned- LACO InLegalBERT

In [ ]:
!pip install -q datasets transformers scikit-learn

import os, re, random, inspect, collections
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModel, AutoModelForQuestionAnswering, TrainingArguments, Trainer, default_data_collator

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = "/kaggle/input/datasets/**/2250-dataset"
TRAIN_FILE = f"{DATA_DIR}/train_dataset.json"
VALID_FILE = f"{DATA_DIR}/validate_dataset.json"
TEST_FILE = f"{DATA_DIR}/test_dataset.json"
OUT_DIR = "/kaggle/working"
MODEL_NAME = "law-ai/InLegalBERT"
BATCH_SIZE = 16
EPOCHS = 4
LEARNING_RATE = 2e-5
MAX_LENGTH = 512
DOC_STRIDE = 128
TOP_RATIO = 0.25
MIN_TOP = 3
MAX_TOP = 8
NEIGHBOUR_WINDOW = 1
MAX_SENT_CHARS = 300
TOKEN_MARGIN = 8

dataset = load_dataset("json", data_files={"train": TRAIN_FILE, "validation": VALID_FILE, "test": TEST_FILE}, field="data")
raw = {name: dataset[name].to_pandas() for name in dataset.keys()}
for name, frame in raw.items():
    frame["answer_start"] = frame["answer"].apply(lambda x: int(x["answer_start"]))
    frame["answer_end"] = frame["answer"].apply(lambda x: int(x["answer_start"]) + len(x["text"]) - 1)
    frame["gold_text"] = [str(c)[s:e + 1] for c, s, e in zip(frame["context"], frame["answer_start"], frame["answer_end"])]
    print(name, len(frame))

SENTENCE_BOUNDARY = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9\"'(])")
SEMICOLON_BOUNDARY = re.compile(r"(?<=;)\s+")

def split_spans(text, start, end, pattern):
    spans = []
    current = start
    for match in pattern.finditer(text, start, end):
        if match.start() > current:
            spans.append((current, match.start()))
        current = match.end()
    if current < end:
        spans.append((current, end))
    return spans

def segment_sentences(text):
    spans = []
    for start, end in split_spans(text, 0, len(text), SENTENCE_BOUNDARY):
        if end - start > MAX_SENT_CHARS:
            spans.extend(split_spans(text, start, end, SEMICOLON_BOUNDARY))
        else:
            spans.append((start, end))
    output = []
    for start, end in spans:
        while start < end and text[start].isspace():
            start += 1
        while end > start and text[end - 1].isspace():
            end -= 1
        if end > start:
            output.append((start, end))
    return output

all_contexts = pd.concat([frame["context"] for frame in raw.values()]).drop_duplicates().tolist()
segment_cache = {context: segment_sentences(context) for context in all_contexts}
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
assert tokenizer.is_fast
embedding_model = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()

@torch.no_grad()
def embed_texts(texts, batch_size=24):
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = list(texts[start:start + batch_size])
        inputs = tokenizer(batch, return_tensors="pt", truncation=True, max_length=512, padding=True).to(device)
        hidden = embedding_model(**inputs).last_hidden_state
        mask = inputs["attention_mask"].unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1)
        pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
        vectors.extend(pooled.cpu().numpy())
    return np.asarray(vectors, dtype=np.float32)

sentence_texts = list(dict.fromkeys(context[start:end] for context in all_contexts for start, end in segment_cache[context]))
sentence_vectors = {}
for start in range(0, len(sentence_texts), 256):
    batch = sentence_texts[start:start + 256]
    sentence_vectors.update(zip(batch, embed_texts(batch).tolist()))
question_vectors = {name: embed_texts(frame["question"].astype(str).tolist()) for name, frame in raw.items()}
embedding_model.to("cpu")
if torch.cuda.is_available():
    torch.cuda.empty_cache()


def optimize_context(question, question_vector, context, spans):
    sentences = [context[start:end] for start, end in spans]
    if not sentences:
        return {"text": context, "kept": [], "top": [], "new_offsets": {}, "scores": []}
    vectors = np.asarray([sentence_vectors[sentence] for sentence in sentences], dtype=np.float32)
    scores = vectors @ question_vector
    token_budget = max(32, MAX_LENGTH - len(tokenizer.tokenize(question)) - TOKEN_MARGIN - 3)
    sentence_lengths = [len(tokenizer.tokenize(sentence)) for sentence in sentences]
    if sum(sentence_lengths) <= token_budget:
        kept = list(range(len(sentences)))
        top = list(range(len(sentences)))
    else:
        count = min(len(sentences), max(MIN_TOP, min(MAX_TOP, int(np.ceil(TOP_RATIO * len(sentences))))))
        top = np.argsort(-scores)[:count].tolist()
        candidates = set(top)
        for index in top:
            for neighbour in range(index - NEIGHBOUR_WINDOW, index + NEIGHBOUR_WINDOW + 1):
                if 0 <= neighbour < len(sentences):
                    candidates.add(neighbour)
        priority = top + sorted(candidates.difference(top), key=lambda index: -scores[index])
        kept = []
        used_tokens = 0
        seen = set()
        for index in priority:
            normalized = re.sub(r"\W+", " ", sentences[index].lower()).strip()
            length = sentence_lengths[index]
            if normalized in seen:
                continue
            if used_tokens + length <= token_budget or not kept:
                kept.append(index)
                used_tokens += length
                seen.add(normalized)
        kept.sort()
    new_offsets = {}
    parts = []
    position = 0
    for index in kept:
        new_offsets[index] = position
        parts.append(sentences[index])
        position += len(sentences[index]) + 1
    return {"text": " ".join(parts), "kept": kept, "top": top, "new_offsets": new_offsets, "scores": scores.tolist()}

def remap_answer(start, end_exclusive, spans, result):
    touched = [i for i, (left, right) in enumerate(spans) if right > start and left < end_exclusive]
    if not touched or any(i not in result["new_offsets"] for i in range(touched[0], touched[-1] + 1)):
        return None
    first, last = touched[0], touched[-1]
    return (result["new_offsets"][first] + max(0, start - spans[first][0]), result["new_offsets"][last] + min(end_exclusive, spans[last][1]) - spans[last][0])

def apply_laco(frame, q_vectors, name):
    rows = []
    for index, row in frame.reset_index(drop=True).iterrows():
        spans = segment_cache[row["context"]]
        result = optimize_context(str(row["question"]), q_vectors[index], row["context"], spans)
        mapped = remap_answer(int(row["answer_start"]), int(row["answer_end"]) + 1, spans, result)
        rows.append({**row.to_dict(), "opt_context": result["text"], "opt_start": mapped[0] if mapped else -1, "opt_end": mapped[1] - 1 if mapped else -1, "covered": mapped is not None, "compression": len(row["context"]) / max(1, len(result["text"]))})
    output = pd.DataFrame(rows)
    print(name, "rows:", len(output), "answer coverage:", round(output["covered"].mean() * 100, 2), "%")
    return output

optimized = {name: apply_laco(frame, question_vectors[name], name) for name, frame in raw.items()}

def make_dataset(frame, training=False):
    if training:
        frame = frame[frame["covered"]].copy()
    ids = frame["id"].astype(str).tolist() if "id" in frame else [str(i) for i in range(len(frame))]
    return Dataset.from_dict({"id": ids, "context": frame["opt_context"].astype(str).tolist(), "question": frame["question"].astype(str).tolist(), "answer": [{"answer_start": int(start), "answer_end": int(end)} for start, end in zip(frame["opt_start"], frame["opt_end"])]})

qa_data = DatasetDict({"train": make_dataset(optimized["train"], True), "validation": make_dataset(optimized["validation"], True), "test": make_dataset(optimized["test"], False)})
print(qa_data)

def prepare_train_features(examples):
    encoded = tokenizer(examples["question"], examples["context"], truncation="only_second", max_length=MAX_LENGTH, stride=DOC_STRIDE, return_overflowing_tokens=True, return_offsets_mapping=True, padding="max_length")
    mapping = encoded.pop("overflow_to_sample_mapping")
    offsets = encoded.pop("offset_mapping")
    starts, ends = [], []
    for i, offset in enumerate(offsets):
        cls_index = encoded["input_ids"][i].index(tokenizer.cls_token_id)
        sequence_ids = encoded.sequence_ids(i)
        answer = examples["answer"][mapping[i]]
        start_char, end_char = answer["answer_start"], answer["answer_end"] + 1
        context_tokens = [j for j, sid in enumerate(sequence_ids) if sid == 1]
        if not context_tokens or offset[context_tokens[0]][0] > start_char or offset[context_tokens[-1]][1] < end_char:
            starts.append(cls_index)
            ends.append(cls_index)
            continue
        start_token = context_tokens[0]
        while start_token < len(offset) and offset[start_token][0] <= start_char:
            start_token += 1
        end_token = context_tokens[-1]
        while end_token >= 0 and offset[end_token][1] >= end_char:
            end_token -= 1
        starts.append(start_token - 1)
        ends.append(end_token + 1)
    encoded["start_positions"] = starts
    encoded["end_positions"] = ends
    return encoded

tokenized_data = DatasetDict({name: split.map(prepare_train_features, batched=True, remove_columns=split.column_names) for name, split in qa_data.items() if name in ["train", "validation"]})
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)
training_parameters = inspect.signature(TrainingArguments.__init__).parameters
evaluation_key = "eval_strategy" if "eval_strategy" in training_parameters else "evaluation_strategy"
training_args = TrainingArguments(output_dir=f"{OUT_DIR}/laco_inlegalbert_qa", **{evaluation_key: "epoch"}, save_strategy="epoch", learning_rate=LEARNING_RATE, per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE, num_train_epochs=EPOCHS, weight_decay=0.01, warmup_ratio=0.1, logging_steps=10, fp16=torch.cuda.is_available(), report_to="none", seed=SEED, load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False)
trainer_parameters = inspect.signature(Trainer.__init__).parameters
tokenizer_key = "processing_class" if "processing_class" in trainer_parameters else "tokenizer"
trainer = Trainer(model=model, args=training_args, train_dataset=tokenized_data["train"], eval_dataset=tokenized_data["validation"], data_collator=default_data_collator, **{tokenizer_key: tokenizer})
trainer.train()
trainer.save_model(f"{OUT_DIR}/LACO_InLegalBERT")
tokenizer.save_pretrained(f"{OUT_DIR}/LACO_InLegalBERT")

def prepare_test_features(examples):
    encoded = tokenizer(examples["question"], examples["context"], truncation="only_second", max_length=MAX_LENGTH, stride=DOC_STRIDE, return_overflowing_tokens=True, return_offsets_mapping=True, padding="max_length")
    mapping = encoded.pop("overflow_to_sample_mapping")
    encoded["example_id"] = []
    for i in range(len(encoded["input_ids"])):
        sequence_ids = encoded.sequence_ids(i)
        encoded["example_id"].append(examples["id"][mapping[i]])
        encoded["offset_mapping"][i] = [value if sequence_ids[j] == 1 else None for j, value in enumerate(encoded["offset_mapping"][i])]
    return encoded

test_features = qa_data["test"].map(prepare_test_features, batched=True, remove_columns=qa_data["test"].column_names)
prediction_features = test_features.remove_columns(["example_id", "offset_mapping"])
predictions_output = trainer.predict(prediction_features)
start_logits, end_logits = predictions_output.predictions
example_to_features = collections.defaultdict(list)
for feature_index, feature in enumerate(test_features):
    example_to_features[feature["example_id"]].append(feature_index)
predictions = {}
for example in qa_data["test"]:
    candidates = []
    for feature_index in example_to_features[example["id"]]:
        offsets = test_features[feature_index]["offset_mapping"]
        start_order = np.argsort(start_logits[feature_index])[-20:][::-1]
        end_order = np.argsort(end_logits[feature_index])[-20:][::-1]
        for start_index in start_order:
            for end_index in end_order:
                if start_index >= len(offsets) or end_index >= len(offsets):
                    continue
                if offsets[start_index] is None or offsets[end_index] is None or end_index < start_index or end_index - start_index + 1 > 100:
                    continue
                candidates.append((start_logits[feature_index][start_index] + end_logits[feature_index][end_index], offsets[start_index][0], offsets[end_index][1]))
    predictions[example["id"]] = max(candidates, key=lambda item: item[0]) if candidates else (0, 0, 0)

def normalize_answer(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return " ".join(text.split())

def token_f1(prediction, reference):
    pred_tokens = normalize_answer(prediction).split()
    ref_tokens = normalize_answer(reference).split()
    if not pred_tokens or not ref_tokens:
        return float(pred_tokens == ref_tokens)
    overlap = sum((collections.Counter(pred_tokens) & collections.Counter(ref_tokens)).values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

results = []
original_test = raw["test"].reset_index(drop=True)
for index, example in enumerate(qa_data["test"]):
    score, start, end = predictions[example["id"]]
    predicted = example["context"][start:end] if end > start else ""
    gold = str(original_test.iloc[index]["gold_text"])
    results.append({"id": example["id"], "question": example["question"], "gold_answer": gold, "predicted_answer": predicted, "exact_match": int(normalize_answer(predicted) == normalize_answer(gold)), "f1": token_f1(predicted, gold)})
results_df = pd.DataFrame(results)
print("Exact Match (%):", round(results_df["exact_match"].mean() * 100, 2))
print("F1-score (%):", round(results_df["f1"].mean() * 100, 2))
results_df.to_csv(f"{OUT_DIR}/laco_inlegalbert_test_results.csv", index=False)
print("Saved model and test results to", OUT_DIR)

def answer_legal_question(question, context):
    spans = segment_sentences(context)
    sentence_list = [context[start:end] for start, end in spans]
    new_texts = [sentence for sentence in sentence_list if sentence not in sentence_vectors]
    if new_texts:
        embedding_model.to(device).eval()
        sentence_vectors.update(zip(new_texts, embed_texts(new_texts).tolist()))
        embedding_model.to("cpu")
    embedding_model.to(device).eval()
    question_vector = embed_texts([question])[0]
    embedding_model.to("cpu")
    optimized = optimize_context(question, question_vector, context, spans)
    inputs = tokenizer(question, optimized["text"], return_tensors="pt", truncation="only_second", max_length=MAX_LENGTH, return_offsets_mapping=True)
    offsets = inputs.pop("offset_mapping")[0].tolist()
    sequence_ids = inputs.sequence_ids(0)
    model_inputs = {key: value.to(trainer.model.device) for key, value in inputs.items()}
    with torch.no_grad():
        output = trainer.model(**model_inputs)
    start_scores = output.start_logits[0].detach().cpu().numpy()
    end_scores = output.end_logits[0].detach().cpu().numpy()
    best = (-1, -1, -float("inf"))
    for start_index in np.argsort(start_scores)[-20:]:
        for end_index in np.argsort(end_scores)[-20:]:
            if end_index < start_index or end_index - start_index + 1 > 100:
                continue
            if sequence_ids[start_index] != 1 or sequence_ids[end_index] != 1:
                continue
            value = start_scores[start_index] + end_scores[end_index]
            if value > best[2]:
                best = (start_index, end_index, value)
    answer = optimized["text"][offsets[best[0]][0]:offsets[best[1]][1]] if best[2] > -float("inf") else ""
    return answer, optimized["text"]

print(results_df.head())